# 03 — LoRA Fine-Tuning of Qwen3.5-35B-A3B

Fine-tune **Qwen/Qwen3.5-35B-A3B** using **LoRA** (Low-Rank Adaptation) via
[PEFT](https://github.com/huggingface/peft) and [TRL SFTTrainer](https://github.com/huggingface/trl).

The MoE architecture (only ~3B params active per step) makes this model
tractable to fine-tune on a single MI250X with BF16 precision.

### What this notebook does
1. Load model in BF16 (no quantization needed thanks to MoE)
2. Attach LoRA adapters to attention and gate projection layers
3. Prepare a small example dataset (instruction-following format)
4. Run SFT training for a configurable number of steps
5. Save the adapter weights
6. Reload and verify the fine-tuned model

**Requirements:** ≥ 1× MI250X (64 GB).

## 0. Imports & Hyperparameters

In [ ]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# ── Hyperparameters ────────────────────────────────────────────────────────
MODEL_ID        = 'Qwen/Qwen3.5-35B-A3B'
OUTPUT_DIR      = '../checkpoints/qwen35-lora'
DTYPE           = torch.bfloat16
DEVICE_MAP      = 'auto'

# LoRA
LORA_R          = 16       # rank
LORA_ALPHA      = 32       # scaling factor (alpha / r = 2)
LORA_DROPOUT    = 0.05
# Target the Gated-Attention layers (linear projections)
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

# Training
MAX_SEQ_LENGTH  = 2048
BATCH_SIZE      = 1        # per device; use gradient accumulation for effective larger batches
GRAD_ACCUM      = 8
LR              = 2e-4
NUM_EPOCHS      = 1
MAX_STEPS       = 50       # set to -1 to run full epochs
WARMUP_RATIO    = 0.1
LOGGING_STEPS   = 5
SAVE_STEPS      = 25
# ───────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Model: {MODEL_ID}')

## 1. Load Tokenizer & Base Model

In [ ]:
print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side='right',   # required for SFT packing
)
# Ensure pad token is defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading model ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE_MAP,
    trust_remote_code=True,
)
model.config.use_cache = False   # disable KV-cache during training
print('Base model loaded.')

## 2. Attach LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: << 1% of total params — LoRA adds tiny adapters

## 3. Prepare Dataset

We use a small synthetic instruction-following dataset to demonstrate the
training loop. Replace `raw_data` with your own examples or load from disk /
HuggingFace Hub.

In [ ]:
# ── Synthetic example data ─────────────────────────────────────────────────
# Each entry follows the instruction-following (chat) format.
# Replace or extend this list with your actual training data.
raw_data = [
    {
        'instruction': 'Explain what a transformer neural network is in simple terms.',
        'response': (
            'A transformer is a type of neural network that processes sequences of data '
            '(like words) using a mechanism called attention. Instead of reading text '
            'word by word in order, it looks at all words simultaneously and learns '
            'which ones are most important for understanding each other. This makes it '
            'very good at tasks like translation, summarization, and question answering.'
        ),
    },
    {
        'instruction': 'Write a Python function to reverse a string.',
        'response': (
            'def reverse_string(s: str) -> str:\n'
            '    """Return the reversed version of the input string."""\n'
            '    return s[::-1]'
        ),
    },
    {
        'instruction': 'What are the main advantages of using a supercomputer like LUMI?',
        'response': (
            'LUMI (Large Unified Modern Infrastructure) offers several key advantages: '
            '(1) Massive GPU compute — thousands of AMD MI250X accelerators for large-scale AI training. '
            '(2) High-bandwidth memory — each MI250X has 128 GB HBM2e for large models. '
            '(3) Fast interconnects — HPE Slingshot fabric reduces communication bottlenecks. '
            '(4) Energy efficiency — powered by hydroelectricity and cooled by district heating. '
            '(5) Open access — available to European researchers via CSC and EuroHPC.'
        ),
    },
    {
        'instruction': 'What is Mixture of Experts (MoE) and why is it useful?',
        'response': (
            'Mixture of Experts (MoE) is a neural network architecture where only a subset '
            'of the model\'s parameters ("experts") are activated for each input token. '
            'A learned router selects which experts to use. This allows the model to have '
            'a very large total parameter count while keeping the compute per forward pass '
            'similar to a much smaller dense model. For example, Qwen3.5-35B-A3B has 35B '
            'total parameters but only activates ~3B per token, giving the knowledge '
            'capacity of a 35B model at the inference cost of a 3B model.'
        ),
    },
    {
        'instruction': 'Summarize the key steps to fine-tune a large language model.',
        'response': (
            '1. Choose a pre-trained base model appropriate for your task.\n'
            '2. Prepare a high-quality task-specific dataset in the model\'s chat format.\n'
            '3. Select a parameter-efficient method such as LoRA to reduce GPU memory.\n'
            '4. Configure training hyperparameters: learning rate, batch size, epochs.\n'
            '5. Train with a framework like TRL SFTTrainer.\n'
            '6. Evaluate on a held-out validation set and iterate.\n'
            '7. Merge adapters or keep them separate for flexible deployment.'
        ),
    },
]

# ── Convert to chat format ──────────────────────────────────────────────────
def to_chat_text(example: dict) -> dict:
    messages = [
        {'role': 'system', 'content': 'You are a helpful AI assistant.'},
        {'role': 'user',   'content': example['instruction']},
        {'role': 'assistant', 'content': example['response']},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,   # fine-tune without thinking traces
    )
    return {'text': text}

dataset = Dataset.from_list(raw_data).map(to_chat_text, remove_columns=raw_data[0].keys())
print(f'Dataset size: {len(dataset)} examples')
print('--- Sample ---')
print(dataset[0]['text'][:400], '...')

## 4. Training

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field='text',
    packing=False,                     # set True to pack short sequences for efficiency

    # Optimization
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    optim='adamw_torch',

    # Precision
    bf16=True,                         # BF16 native training
    fp16=False,
    gradient_checkpointing=True,       # saves VRAM at cost of ~20% slower step

    # Schedule
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,               # overrides epochs if > 0

    # Logging & saving
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to='none',                  # change to 'wandb' or 'tensorboard' if desired
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

print('Starting training ...')
trainer.train()
print('Training complete.')

## 5. Save LoRA Adapter

In [ ]:
adapter_path = os.path.join(OUTPUT_DIR, 'final_adapter')
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Adapter saved to: {adapter_path}')

import os as _os
saved_files = _os.listdir(adapter_path)
print('Saved files:', saved_files)

## 6. Reload & Verify Fine-Tuned Model

In [ ]:
from peft import PeftModel

print('Reloading base model ...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE_MAP,
    trust_remote_code=True,
)

print('Attaching fine-tuned LoRA adapter ...')
ft_model = PeftModel.from_pretrained(base_model, adapter_path)
ft_model.eval()
print('Fine-tuned model ready.')

In [ ]:
def ft_generate(prompt: str, max_new_tokens: int = 256) -> str:
    messages = [
        {'role': 'system', 'content': 'You are a helpful AI assistant.'},
        {'role': 'user',   'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors='pt').to(ft_model.device)
    with torch.no_grad():
        out_ids = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

# Test on a prompt seen during training
result = ft_generate('What is Mixture of Experts (MoE) and why is it useful?')
print('=== Fine-tuned model response ===')
print(result)

In [ ]:
# Test on an unseen prompt to check generalisation
result2 = ft_generate('How does gradient checkpointing help with GPU memory?')
print('=== Generalisation test ===')
print(result2)

## 7. Training Loss Plot

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps  = [e['step'] for e in log_history if 'loss' in e]
losses = [e['loss'] for e in log_history if 'loss' in e]

if steps:
    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses, marker='o', linewidth=2)
    plt.xlabel('Step')
    plt.ylabel('Training Loss')
    plt.title('Qwen3.5-35B-A3B LoRA Fine-Tuning Loss')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curve.png'), dpi=150)
    plt.show()
    print(f'Final loss: {losses[-1]:.4f}')
else:
    print('No loss history recorded (training may not have run).')

---
### Next Steps

- **Scale up data**: replace `raw_data` with a larger instruction dataset (e.g. from HF Hub)
- **Increase steps / epochs**: set `MAX_STEPS=-1` and `NUM_EPOCHS=3` for full training
- **Merge adapter**: use `model.merge_and_unload()` to fuse LoRA weights into the base model for deployment
- **Evaluate**: use `lm-evaluation-harness` or a task-specific benchmark
- **Multi-GPU**: set `DEVICE_MAP='auto'` and request multiple MI250X nodes in SLURM